# Factor and Style Classification for Recommendation and Retrieval Datasets

This notebook runs two inference stages:

1. Factor classification
   - category
   - sub_category
   - color
   - material
   - pattern

2. Style classification
   - style

For each output seed:
- The corresponding retrieval dataset and `recom_dataset` are predicted using the same factor model weights.
- The style model is shared across all seeds.

Output files will be saved to:
`classification_results/seed_1`, `classification_results/seed_2`, ..., `classification_results/seed_5`

In [1]:
import os
import json
from pathlib import Path
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from tqdm import tqdm

import clip

# =========================================================
# Basic path configuration
# =========================================================

EXP_ROOT = Path(".").resolve()
PROJECT_ROOT = EXP_ROOT.parent.parent
DATA_ROOT = PROJECT_ROOT / "1_data"

OUTPUT_ROOT = EXP_ROOT / "classification_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Retrieval datasets
RETRIEVAL_ROOT = DATA_ROOT
RETRIEVAL_SEED_MAP = {
    "seed_1": "seed_11",
    "seed_2": "seed_22",
    "seed_3": "seed_33",
    "seed_4": "seed_44",
    "seed_5": "seed_55",
}

# Query dataset
RECOM_DATASET_DIR = DATA_ROOT / "recom_dataset"

# Factor model weights
FACTOR_MODEL_ROOT = DATA_ROOT / "TwoStage_LP_results_colab"

# Style model checkpoint
STYLE_CKPT_PATH = EXP_ROOT / "ft_result" / "openfashionclip_partial_ft" / "checkpoints" / "best.pt"

# Label library for factor prediction
LABEL_PATH = DATA_ROOT / "label_library.json"

# Image extensions
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tiff"}

# Inference settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
NUM_WORKERS = 0
MULTI_LABEL_THRESHOLD = 0.5

print("EXP_ROOT:", EXP_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)

EXP_ROOT: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318
PROJECT_ROOT: C:\Users\Sandy\OneDrive\桌面\Thesis
DATA_ROOT: C:\Users\Sandy\OneDrive\桌面\Thesis\1_data
OUTPUT_ROOT: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results
DEVICE: cuda


In [2]:
# =========================================================
# Load factor label library
# =========================================================

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    label_lib = json.load(f)

single_attrs = ["category", "sub_category"]
multi_attrs = ["color", "pattern", "material"]

print("Loaded factor labels:")
for k in single_attrs + multi_attrs:
    print(f"{k}: {len(label_lib[k])}")

# =========================================================
# Style label mapping
# =========================================================

style_to_idx = {
    "Casual": 0,
    "Ethnic": 1,
    "Formal": 2,
    "Sports": 3,
    "conservative": 4,
    "dressy": 5,
    "fairy": 6,
    "feminine": 7,
    "gal": 8,
    "girlish": 9,
    "kireime-casual": 10,
    "lolita": 11,
    "mode": 12,
    "natural": 13,
    "retro": 14,
    "rock": 15,
    "street": 16,
}

idx_to_style = {v: k for k, v in style_to_idx.items()}
NUM_STYLE_CLASSES = len(style_to_idx)

print("style:", NUM_STYLE_CLASSES)

Loaded factor labels:
category: 10
sub_category: 99
color: 2654
pattern: 8162
material: 2826
style: 17


In [ ]:
# =========================================================
# Utility functions
# =========================================================

def safe_torch_load(path, map_location="cpu"):
    return torch.load(path, map_location=map_location)

def strip_prefix_if_needed(state_dict, prefixes=("module.", "model.", "backbone.")):
    new_state = {}
    for k, v in state_dict.items():
        new_k = k
        for p in prefixes:
            if new_k.startswith(p):
                new_k = new_k[len(p):]
        new_state[new_k] = v
    return new_state

def get_all_image_paths(folder: Path) -> List[Path]:
    image_paths = []
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            image_paths.append(p)
    return sorted(image_paths)

def image_id_from_path(path: Path) -> str:
    return path.stem

def join_multilabel(labels: List[str]) -> str:
    if not labels:
        return ""
    return "|".join(labels)

# =========================================================
# Dataset for inference
# =========================================================

class ImageInferenceDataset(Dataset):
    def __init__(self, image_paths: List[Path], preprocess):
        self.image_paths = image_paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        image_tensor = self.preprocess(image)
        return {
            "image": image_tensor,
            "image_path": str(image_path),
            "image_id": image_id_from_path(image_path),
        }

def collate_fn(batch):
    images = torch.stack([x["image"] for x in batch], dim=0)
    image_paths = [x["image_path"] for x in batch]
    image_ids = [x["image_id"] for x in batch]
    return {
        "images": images,
        "image_paths": image_paths,
        "image_ids": image_ids,
    }

## Factor model

In [ ]:
def load_factor_model_for_seed_dir(seed_dir_name: str):

    seed_num = seed_dir_name.split("_")[-1]

    backbone_path = FACTOR_MODEL_ROOT / seed_dir_name / "best_backbone" / f"clip_visual_color_tuned_seed_{seed_num}.pt"
    head_path = FACTOR_MODEL_ROOT / seed_dir_name / "best_head" / f"linear_probe_best_heads_seed_{seed_num}.pt"

    if not backbone_path.exists():
        raise FileNotFoundError(f"Missing backbone checkpoint: {backbone_path}")
    if not head_path.exists():
        raise FileNotFoundError(f"Missing head checkpoint: {head_path}")

    model, preprocess = clip.load("ViT-B/32", device=DEVICE)

    # Load tuned visual backbone
    ckpt_backbone = safe_torch_load(backbone_path, map_location="cpu")
    visual_state = ckpt_backbone["visual"] if isinstance(ckpt_backbone, dict) and "visual" in ckpt_backbone else ckpt_backbone
    visual_state = strip_prefix_if_needed(visual_state)
    model.visual.load_state_dict(visual_state, strict=False)

    for p in model.parameters():
        p.requires_grad = False
    model.eval()

    feat_dim = model.visual.output_dim

    heads = nn.ModuleDict({
        "single": nn.ModuleDict({
            a: nn.Linear(feat_dim, len(label_lib[a])) for a in single_attrs
        }),
        "multi": nn.ModuleDict({
            a: nn.Linear(feat_dim, len(label_lib[a])) for a in multi_attrs
        }),
    }).to(DEVICE)

    ckpt_heads = safe_torch_load(head_path, map_location="cpu")
    state_dict = ckpt_heads["state_dict"] if isinstance(ckpt_heads, dict) and "state_dict" in ckpt_heads else ckpt_heads
    state_dict = strip_prefix_if_needed(state_dict)
    heads.load_state_dict(state_dict, strict=True)
    heads.eval()

    return model, preprocess, heads

## Style Model

In [27]:
# =========================================================
# Style model definition
# =========================================================
# This implementation assumes:
# - CLIP ViT-B/32 visual encoder
# - A linear classifier on top of image features
#
# If your style training architecture is slightly different,
# only this cell should need adjustment.
# =========================================================

class CLIPStyleClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.clip_model, _ = clip.load("ViT-B/32", device="cpu")
        feat_dim = self.clip_model.visual.output_dim
        self.classifier = nn.Linear(feat_dim, num_classes)

    def encode_image(self, images):
        feats = self.clip_model.encode_image(images)
        return feats.float()

    def forward(self, images):
        feats = self.encode_image(images)
        logits = self.classifier(feats)
        return logits

def load_style_model():
    if not STYLE_CKPT_PATH.exists():
        raise FileNotFoundError(f"Missing style checkpoint: {STYLE_CKPT_PATH}")

    model = CLIPStyleClassifier(num_classes=NUM_STYLE_CLASSES)

    ckpt = safe_torch_load(STYLE_CKPT_PATH, map_location="cpu")

    # Try common checkpoint formats
    candidate_states = []
    if isinstance(ckpt, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                candidate_states.append(ckpt[key])
        candidate_states.append(ckpt)
    else:
        candidate_states.append(ckpt)

    loaded = False
    errors = []

    for state in candidate_states:
        try:
            state = strip_prefix_if_needed(state, prefixes=("module.",))
            missing, unexpected = model.load_state_dict(state, strict=False)
            print("Style model loaded with strict=False")
            print("Missing keys:", missing)
            print("Unexpected keys:", unexpected)
            loaded = True
            break
        except Exception as e:
            errors.append(str(e))

    if not loaded:
        raise RuntimeError("Failed to load style checkpoint.\n" + "\n".join(errors))

    model = model.to(DEVICE)
    model.eval()
    return model

## Data Collection Functions

In [28]:
# =========================================================
# Collect images for recom_dataset and retrieval dataset
# =========================================================

def collect_recom_dataset_images() -> List[Path]:
    if not RECOM_DATASET_DIR.exists():
        raise FileNotFoundError(f"Missing recom_dataset directory: {RECOM_DATASET_DIR}")
    image_paths = get_all_image_paths(RECOM_DATASET_DIR)
    print(f"Found {len(image_paths)} images in recom_dataset")
    return image_paths

def collect_retrieval_split_images(seed_dir_name: str) -> Dict[str, List[Path]]:
    """
    Input example:
    seed_dir_name = 'seed_11'

    Expected structure:
    seed_11/
        train/images
        valid/images
        test/images
    """
    seed_root = RETRIEVAL_ROOT / seed_dir_name
    if not seed_root.exists():
        raise FileNotFoundError(f"Missing retrieval dataset folder: {seed_root}")

    split_map = {
        "train": seed_root / "train" / "images",
        "val": seed_root / "valid" / "images",
        "test": seed_root / "test" / "images",
    }

    result = {}
    for split_name, split_dir in split_map.items():
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing split image folder: {split_dir}")
        result[split_name] = get_all_image_paths(split_dir)
        print(f"{seed_dir_name} - {split_name}: {len(result[split_name])} images")

    return result

## Factor Inference

In [29]:
# =========================================================
# Factor inference
# =========================================================

@torch.no_grad()
def predict_factors(image_paths: List[Path], factor_model, factor_preprocess, factor_heads,
                    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                    threshold=MULTI_LABEL_THRESHOLD) -> pd.DataFrame:

    dataset = ImageInferenceDataset(image_paths, factor_preprocess)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if DEVICE == "cuda" else False,
    )

    rows = []

    for batch in tqdm(loader, desc="Factor inference"):
        images = batch["images"].to(DEVICE)
        image_ids = batch["image_ids"]

        image_feats = factor_model.encode_image(images).float()

        single_preds = {}
        for attr in single_attrs:
            logits = factor_heads["single"][attr](image_feats)
            pred_idx = torch.argmax(logits, dim=1).cpu().tolist()
            single_preds[attr] = [label_lib[attr][i] for i in pred_idx]

        multi_preds = {}
        for attr in multi_attrs:
            logits = factor_heads["multi"][attr](image_feats)
            probs = torch.sigmoid(logits).cpu()

            attr_labels_batch = []
            for i in range(probs.shape[0]):
                selected_idx = torch.where(probs[i] >= threshold)[0].tolist()

                # Fallback to top-1 if nothing passes threshold
                if len(selected_idx) == 0:
                    selected_idx = [int(torch.argmax(probs[i]).item())]

                selected_labels = [label_lib[attr][j] for j in selected_idx]
                attr_labels_batch.append(join_multilabel(selected_labels))

            multi_preds[attr] = attr_labels_batch

        for i in range(len(image_ids)):
            row = {
                "image_id": image_ids[i],
                "category": single_preds["category"][i],
                "sub_category": single_preds["sub_category"][i],
                "color": multi_preds["color"][i],
                "material": multi_preds["material"][i],
                "pattern": multi_preds["pattern"][i],
            }
            rows.append(row)

    return pd.DataFrame(rows)

## Style Inference

In [30]:
# =========================================================
# Style inference
# =========================================================

@torch.no_grad()
def predict_style(image_paths: List[Path], style_model,
                  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS) -> pd.DataFrame:

    # Reuse standard CLIP preprocess
    _, style_preprocess = clip.load("ViT-B/32", device="cpu")

    dataset = ImageInferenceDataset(image_paths, style_preprocess)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if DEVICE == "cuda" else False,
    )

    rows = []

    for batch in tqdm(loader, desc="Style inference"):
        images = batch["images"].to(DEVICE)
        image_ids = batch["image_ids"]

        logits = style_model(images)
        pred_idx = torch.argmax(logits, dim=1).cpu().tolist()
        pred_labels = [idx_to_style[i] for i in pred_idx]

        for img_id, style_label in zip(image_ids, pred_labels):
            rows.append({
                "image_id": img_id,
                "style": style_label,
            })

    return pd.DataFrame(rows)

## Merge factor and style predictions

In [31]:
def merge_factor_and_style(factor_df: pd.DataFrame, style_df: pd.DataFrame) -> pd.DataFrame:
    merged = factor_df.merge(style_df, on="image_id", how="inner")

    final_cols = [
        "image_id",
        "category",
        "sub_category",
        "color",
        "material",
        "pattern",
        "style",
    ]
    merged = merged[final_cols].copy()
    return merged

def save_prediction_csv(df: pd.DataFrame, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path, index=False, encoding="utf-8")
    print(f"Saved: {save_path} | rows = {len(df)}")

## Run All seeds

In [32]:
# =========================================================
# Run full prediction for one output seed
# =========================================================

def run_one_seed(output_seed_name: str, retrieval_seed_dir_name: str, style_model):
    """
    Example:
    output_seed_name = 'seed_1'
    retrieval_seed_dir_name = 'seed_11'
    """

    print("=" * 80)
    print(f"Running output seed: {output_seed_name}")
    print(f"Using retrieval dataset: {retrieval_seed_dir_name}")
    print("=" * 80)

    seed_output_dir = OUTPUT_ROOT / output_seed_name
    seed_output_dir.mkdir(parents=True, exist_ok=True)

    # Load factor model for this seed
    factor_model, factor_preprocess, factor_heads = load_factor_model_for_seed_dir(retrieval_seed_dir_name)

    # Collect images
    recom_images = collect_recom_dataset_images()
    retrieval_splits = collect_retrieval_split_images(retrieval_seed_dir_name)

    # -------------------------
    # recom_dataset
    # -------------------------
    print(f"\n[{output_seed_name}] Predicting recom_dataset")
    factor_df = predict_factors(recom_images, factor_model, factor_preprocess, factor_heads)
    style_df = predict_style(recom_images, style_model)
    merged_df = merge_factor_and_style(factor_df, style_df)
    save_prediction_csv(merged_df, seed_output_dir / "recom_dataset.csv")

    # -------------------------
    # train / val / test
    # -------------------------
    for split_name in ["train", "val", "test"]:
        print(f"\n[{output_seed_name}] Predicting split: {split_name}")
        split_images = retrieval_splits[split_name]

        factor_df = predict_factors(split_images, factor_model, factor_preprocess, factor_heads)
        style_df = predict_style(split_images, style_model)
        merged_df = merge_factor_and_style(factor_df, style_df)

        save_prediction_csv(merged_df, seed_output_dir / f"{split_name}.csv")


# =========================================================
# Main execution
# =========================================================

print("Loading shared style model...")
style_model = load_style_model()

for output_seed_name, retrieval_seed_dir_name in RETRIEVAL_SEED_MAP.items():
    run_one_seed(
        output_seed_name=output_seed_name,
        retrieval_seed_dir_name=retrieval_seed_dir_name,
        style_model=style_model,
    )

print("\nAll seeds finished.")

Loading shared style model...


C:\Users\Sandy\AppData\Local\Temp\ipykernel_21788\1317973752.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location=map_location)


Style model loaded with strict=False
Missing keys: ['clip_model.positional_embedding', 'clip_model.text_projection', 'clip_model.logit_scale', 'clip_model.visual.class_embedding', 'clip_model.visual.positional_embedding', 'clip_model.visual.proj', 'clip_model.visual.conv1.weight', 'clip_model.visual.ln_pre.weight', 'clip_model.visual.ln_pre.bias', 'clip_model.visual.transformer.resblocks.0.attn.in_proj_weight', 'clip_model.visual.transformer.resblocks.0.attn.in_proj_bias', 'clip_model.visual.transformer.resblocks.0.attn.out_proj.weight', 'clip_model.visual.transformer.resblocks.0.attn.out_proj.bias', 'clip_model.visual.transformer.resblocks.0.ln_1.weight', 'clip_model.visual.transformer.resblocks.0.ln_1.bias', 'clip_model.visual.transformer.resblocks.0.mlp.c_fc.weight', 'clip_model.visual.transformer.resblocks.0.mlp.c_fc.bias', 'clip_model.visual.transformer.resblocks.0.mlp.c_proj.weight', 'clip_model.visual.transformer.resblocks.0.mlp.c_proj.bias', 'clip_model.visual.transformer.resbl

Style inference: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]


Saved: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_1\recom_dataset.csv | rows = 50

[seed_1] Predicting split: train


...

Saved: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_5\train.csv | rows = 28792

[seed_5] Predicting split: val


Style inference: 100%|██████████| 57/57 [00:12<00:00,  4.66it/s]


Saved: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_5\val.csv | rows = 3599

[seed_5] Predicting split: test


Style inference: 100%|██████████| 57/57 [00:12<00:00,  4.69it/s]

Saved: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_5\test.csv | rows = 3599

All seeds finished.


In [34]:
# Add a description column to produced CSVs using the template:
# "A {color} {material} {pattern} {sub_category} {category} in a {style} style."
CSV_FILES = ["recom_dataset.csv", "train.csv", "val.csv", "test.csv"]

def _clean_val(v):
    if pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s

def _make_description(row):
    parts = []
    for k in ["color", "material", "pattern", "sub_category", "category"]:
        v = _clean_val(row.get(k, ""))
        if v:
            parts.append(v)
    phrase = " ".join(parts).strip()
    style = _clean_val(row.get("style", ""))
    if phrase and style:
        return f"A {phrase} in a {style} style."
    if phrase:
        return f"A {phrase}."
    if style:
        return f"A {style} style."
    return ""

for output_seed_name in RETRIEVAL_SEED_MAP.keys():
    seed_dir = OUTPUT_ROOT / output_seed_name
    for file_name in CSV_FILES:
        csv_path = seed_dir / file_name
        if not csv_path.exists():
            continue
        df_in = pd.read_csv(csv_path)
        df_in["description"] = df_in.apply(_make_description, axis=1)
        df_in.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Updated: {csv_path} | rows = {len(df_in)}")

Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_1\recom_dataset.csv | rows = 50
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_1\train.csv | rows = 28792
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_1\val.csv | rows = 3599
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_1\test.csv | rows = 3599
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_2\recom_dataset.csv | rows = 50
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_2\train.csv | rows = 28792
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_2\val.csv | rows = 3599
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_2\test.csv | rows = 3599
Updated: C:\Users\Sandy\OneDrive\桌面\Thesis\1_exp\0318\classification_results\seed_3\recom_dataset.csv | rows = 50
Updated: C:\Us

## Summary check

In [35]:
for output_seed_name in RETRIEVAL_SEED_MAP.keys():
    seed_dir = OUTPUT_ROOT / output_seed_name
    print(f"\nSummary for {output_seed_name}")
    for file_name in ["recom_dataset.csv", "train.csv", "val.csv", "test.csv"]:
        csv_path = seed_dir / file_name
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"{file_name}: {len(df)} rows")
        else:
            print(f"{file_name}: missing")


Summary for seed_1
recom_dataset.csv: 50 rows
train.csv: 28792 rows
val.csv: 3599 rows
test.csv: 3599 rows

Summary for seed_2
recom_dataset.csv: 50 rows
train.csv: 28792 rows
val.csv: 3599 rows
test.csv: 3599 rows

Summary for seed_3
recom_dataset.csv: 50 rows
train.csv: 28792 rows
val.csv: 3599 rows
test.csv: 3599 rows

Summary for seed_4
recom_dataset.csv: 50 rows
train.csv: 28792 rows
val.csv: 3599 rows
test.csv: 3599 rows

Summary for seed_5
recom_dataset.csv: 50 rows
train.csv: 28792 rows
val.csv: 3599 rows
test.csv: 3599 rows
